**Day 4 — Advanced PySpark Transformations + Window Functions 🚀**

**Q1 — Explode an Array
🏢 Business Problem**

An e-commerce company stores all products purchased in an order inside an array.

For example:

`O101 → [Laptop, Mouse, Keyboard]
O102 → [Mobile, Charger]`

The analytics team wants one row per product so that product-level analysis can be performed.

Data

Create this DataFrame:

`data = [
    ("O101", "C001", ["Laptop", "Mouse", "Keyboard"]),
    ("O102", "C002", ["Mobile", "Charger"]),
    ("O103", "C001", ["Tablet"]),
    ("O104", "C003", ["Laptop", "Mouse"])
]
`
`columns = ["order_id", "customer_id", "products"]`

Create:

df
- 🎯 Task

Convert the array into individual rows using explode().

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("day4").getOrCreate()
import pyspark.sql.functions as F

In [0]:
data = [
    ("O101", "C001", ["Laptop", "Mouse", "Keyboard"]),
    ("O102", "C002", ["Mobile", "Charger"]),
    ("O103", "C001", ["Tablet"]),
    ("O104", "C003", ["Laptop", "Mouse"])
]

columns = ["order_id", "customer_id", "products"]

df=spark.createDataFrame(data,columns)
df.show()

In [0]:
explode_df=df.withColumn("product",F.explode("products"))
explode_df.show()

Day 4 — Q2: Explode + Aggregation

Now let's make it a real business problem.

🏢 Business Problem

The product team wants to know:

How many times was each product included across all orders?

Using the exploded DataFrame from Q1, calculate the number of orders/products for each product.

In [0]:
explode_df.groupBy("product").agg(
    F.count("product").alias("order_count")
).show()

Q3 — Structs / Nested Data
🏢 Business Problem

Your orders now contain customer information as a nested structure.

Create this DataFrame:



`data = [
    ("O101", ("C001", "Shivam", "Delhi"), 1500),
    ("O102", ("C002", "Rahul", "Mumbai"), 800),
    ("O103", ("C003", "Priya", "Bangalore"), 1200),
]`

`columns = ["order_id", "customer", "amount"]`

The customer column should contain:

 customer
-  ├─ customer_id
-  ├─ name
-  └─ city

🎯 Task

Create a proper StructType schema for the customer field and create the DataFrame.

Then extract:

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
data = [
    ("O101", ("C001", "Shivam", "Delhi"), 1500),
    ("O102", ("C002", "Rahul", "Mumbai"), 800),
    ("O103", ("C003", "Priya", "Bangalore"), 1200),
]

columns = ["order_id", "customer", "amount"]

customer_schema=StructType([
    StructField("customer_id",StringType(), True),
    StructField("name",StringType(),True),
    StructField("city",StringType(),True)
])

schema=StructType([
    StructField("order_id",StringType(),True),
     StructField("customer",customer_schema,True),
     StructField("amount",IntegerType(),True)
])

df=spark.createDataFrame(data,schema)
df.show()


Q3 — Part 2: Extract Nested Fields

Now the actual business requirement:

The analytics team doesn't want the nested customer struct in the final reporting table.


In [0]:
df.select("order_id","customer.customer_id","customer.name","customer.city","amount").show()

Day 4 — Q4: Date Functions
🏢 Business Problem

An e-commerce company wants to analyze order delivery performance.

You have:

 data = [
-     ("O101", "2026-08-01", "2026-08-05"),
-     ("O102", "2026-08-03", "2026-08-10"),
-     ("O103", "2026-08-05", "2026-08-06"),
-     ("O104", "2026-08-10", "2026-08-15"), 
]

- columns = ["order_id", "order_date", "delivery_date"]

Create the DataFrame.

🎯 Task

Create a new column:

delivery_days

which tells us how many days it took to deliver each order.

In [0]:
import pyspark.sql.functions as F

In [0]:
data = [
    ("O101", "2026-08-01", "2026-08-05"),
    ("O102", "2026-08-03", "2026-08-10"),
    ("O103", "2026-08-05", "2026-08-06"),
    ("O104", "2026-08-10", "2026-08-15"),
]

columns = ["order_id", "order_date", "delivery_date"]

df2=spark.createDataFrame(data,columns)
df2.show()

In [0]:
df2.select("order_id",F.to_date("order_date").alias("order_date"),"delivery_date",F.datediff("delivery_date","order_date").alias("delivery_days")).show()

Q5 — Date Extraction

🏢 Business Problem
The business team wants to analyze orders by year, month, and day.
Using the same df2, create three new columns:
- order_year
- order_month
- order_day

In [0]:
df2.select("order_id",F.year("order_date").alias("year"),F.month("order_date").alias("month"), F.day("order_date").alias("day")).show()

Q6 — Date Arithmetic
🏢 Business Problem

The logistics team promises customers:
Delivery should happen within 5 days of the order date.

In [0]:
df2.select("order_id","order_date","delivery_date",F.date_add("order_date",5).alias("expected_delivery_date"),
           F.when(F.col("delivery_date")<=F.col("expected_delivery_date"),"On-time").otherwise("Late").alias("status")).show()

Q7 — String Transformation + regexp_replace()
🏢 Business Problem

Your customer source system contains messy customer names:

In [0]:
customer_data = [
    ("C001", "  Shivam  Mishra  "),
    ("C002", "Rahul-Kumar"),
    ("C003", " Priya@Sharma "),
    ("C004", "Amit  Singh"),
]

columns = ["customer_id", "customer_name"]
df=spark.createDataFrame(customer_data,columns)
df.show()

In [0]:
df2=df.withColumn("customer_clean_name",F.regexp_replace("customer_name","[-,@]"," "))
df2.select("customer_id",F.trim("customer_clean_name").alias("customer_clean_name")).show()

Day 4 — Q8: concat() 🔥

🏢 Business Problem
The analytics team wants a customer display name for dashboards.

In [0]:
customer_data = [
    ("C001", "Shivam", "Mishra"),
    ("C002", "Rahul", "Kumar"),
    ("C003", "Priya", "Sharma"),
    ("C004", "Amit", "Singh")
]

columns = ["customer_id", "first_name", "last_name"]

df = spark.createDataFrame(customer_data, columns)

In [0]:
df2=df.withColumn("customer_display_name",F.concat(F.col("first_name"),F.lit(" "),F.col("last_name")))
df2.show()

Q9 — explode() + Aggregation
🏢 Business Problem
You have orders where each order can contain multiple products.


The products column is an array.
🎯 Business requirement
The analytics team wants to know:
How many orders contain each product?

In [0]:
data = [
    ("O101", "C001", ["Laptop", "Mouse", "Keyboard"]),
    ("O102", "C002", ["Mobile", "Charger"]),
    ("O103", "C001", ["Laptop", "Mouse"]),
    ("O104", "C003", ["Tablet", "Cover"]),
]

columns = ["order_id", "customer_id", "products"]

df=spark.createDataFrame(data, columns)
df.show()
explode_df=df.withColumn("product",F.explode("products"))
explode_df.show()
explode_df.groupBy("product").agg(
    F.count("order_id").alias("order count")
).show()

Q10 — struct extraction
Now let's move to STRUCT, since you just created a nested customer struct earlier.
Business problem:
Your DataFrame has:

In [0]:
data = [
    ("O101", ("C001", "Shivam", "Delhi"), 1500),
    ("O102", ("C002", "Rahul", "Mumbai"), 800),
    ("O103", ("C003", "Priya", "Bangalore"), 1200),
]

columns = ["order_id", "customer", "amount"]



customer_schema=StructType([
    StructField("customer_id",StringType(),True),
    StructField("name",StringType(),True),
    StructField("city",StringType(),True)]
)
column_schema=StructType([
    StructField("order_id",StringType(),True),
    StructField("customer",customer_schema,True),
    StructField("amount",IntegerType(),True)
])

df=spark.createDataFrame(data,column_schema)
df.show()
df.select("order_id","customer.customer_id","customer.name","customer.city","amount").show()

In [0]:
df.select("order_id","customer.customer_id","customer.name","customer.city","amount").show()

Q11 — Window Function: lag()
Now let's move into the window functions part of Day 4.
Business problem:
The analytics team wants to compare each customer's current order amount with their previous order amount.

In [0]:
data = [
    ("C001", "O101", "2026-08-01", 1500),
    ("C001", "O102", "2026-08-05", 800),
    ("C001", "O103", "2026-08-10", 1200),
    ("C002", "O104", "2026-08-02", 2000),
    ("C002", "O105", "2026-08-08", 500),
]

columns=["customer","order_id","order_date","amount"]
df=spark.createDataFrame(data, columns)
df.show()


In [0]:
from pyspark.sql.window import Window

In [0]:
wind=Window.partitionBy("customer").orderBy(F.col("order_date").desc())
df2=df.withColumn("previous_amount",F.lag("amount").over(wind))
df2.show()

